# Описание полученного варианта

## Вариант 3 (№21 в списке): 
- Исследуемые поля:
    - timestamp
    - hour
    - is_weekend
    - device_type
    - os_family
    - browser_family
    - known_device
- Основная проверка:
Временное разделение по timestamp

In [1]:
# Подготовка проекта

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
DATA_PATH = Path('data/auth_events_lab01.xlsx')

print(sys.version)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

3.11.16 (main, Aug 13 2026, 09:46:36) [GCC 15.2.0]
pandas: 2.2.2
scikit-learn: 1.4.2


## Постановка задачи и паспорт датасета

### Прикладная задача ИБ

Требуется оценить, является ли конкретная попытка аутентификации подозрительной, используя контекст устройства и времени события (тип устройства, семейство ОС, семейство браузера, признак известного устройства, час суток, признак выходного дня). Результат предназначен для аналитика SOC / системы мониторинга ИБ: он позволяет приоритизировать проверки, повысить уровень логирования для подозрительных сессий, запросить дополнительный фактор аутентификации, ограничить доступ к чувствительным ресурсам или инициировать ручную верификацию события. Автоматическое блокирование учетной записи по одной метке не предполагается — решение носит характер поддержки принятия решений.

### Объект анализа и гранулярность наблюдения

**Объект анализа** — одна попытка аутентификации пользователя.

**Гранулярность наблюдения** — одна строка таблицы соответствует ровно одной попытке входа, зафиксированной средством защиты. 

Одна строка не соответствует одному пользователю или инциденту, так как:

- один пользователь за период наблюдения совершает много попыток, поэтому строка на пользователя потеряла бы событийный контекст;
- один инцидент может объединять несколько попыток (например, серию неуспешных входов с последующим успехом), поэтому строка на инцидент смешала бы разные моменты принятия решения;
- решение принимается **в момент попытки**, поэтому единица наблюдения — именно попытка.

### Целевая переменная `is_suspicious`

**Целевая переменная:** `is_suspicious` — бинарная метка, отражающая итоговую оценку события.

- `1` — событие признано **подозрительным** (повышенный риск);
- `0` — событие признано **нормальным** (риск в пределах допустимого).

Имя переменной `is_suspicious` не является аргументом в пользу корректности разметки. Оно задает только семантику целевого признака — заявленный смысл "подозрительное / нормальное событие", — но не подтверждает, что проставленные в наборе метки действительно соответствуют реальному классу объекта.

Корректность разметки должна обосновываться отдельно и независимо от названия столбца: через источник меток (кто или что их формирует) и через процедуру их получения (по какому правилу и в какой момент). Пока источник и процедура не проверены, имя `is_suspicious` остается лишь гипотезой о смысле признака, а не доказательством качества метки.

### Паспорт датасета

**Источник:** лист «Данные», из таблицы, прикрепленной к лабораторной работе (https://disk.yandex.ru/i/jROt0cPOXj_qHA).

**Дата получения:** 23.09.2026.

**Исходный размер:** 163 Kb

**Ограничения:** 

Набор синтетически сгенерированный, поэтому существует ряд ограничений:

- распределения признаков и метки могут не соответствовать реальному SOC;
- источник не содержит реальных персональных, конфиденциальных или опасных данных;
- правила генерации метки и признаков неизвестны, поэтому возможно наличие скрытых зависимостей между полями;
- часть полей (`analyst_verdict`, `analyst_risk_score`) появляется **после** события и не должна использоваться при принятии решения.

### Таблица признаков

| Столбец | Смысл | Фактический тип | Ожидаемый тип | Допустимые значения | Момент доступности | Решение |
|---|---|---|---|---|---|---|
| `event_id` | Технический идентификатор записи | string | string | уникальная строка | до события (служебный) | исключить |
| `timestamp` | Время попытки аутентификации | datetime | datetime | в пределах периода сбора | в момент события | использовать только для разделения |
| `user_id` | Псевдоним учетной записи | category | category | `usr_XXX` | в момент события | использовать как группу при анализе, из X исключить |
| `department` | Подразделение пользователя | category | category | IT, HR, SOC, R&D, Sales, Finance, Operations | в момент события | оставить как контекст |
| `source_ip` | IP-адрес источника | string | string | IPv4 | в момент события | оставить как контекст |
| `country` | Страна по IP | category | category | ISO-код | в момент события | оставить как контекст |
| `device_type` | Тип устройства | category | category | desktop, laptop, tablet, mobile | в момент события | оставить |
| `os_family` | Семейство ОС | category | category | Windows, macOS, Linux, iOS, Android | в момент события | оставить |
| `browser_family` | Семейство браузера | category | category | Chrome, Firefox, Safari, Edge | в момент события | оставить |
| `auth_method` | Способ аутентификации | category | category | password, otp, push, fido2 | в момент события | оставить как контекст |
| `hour` | Час суток по UTC | integer | integer | 0...23 | в момент события | оставить |
| `is_weekend` | Признак выходного дня | binary | binary | {0, 1} | в момент события | оставить |
| `failed_attempts_24h` | Неуспешных попыток за 24 ч | integer | integer | >= 0 | в момент события | оставить как контекст |
| `login_velocity_1h` | Попыток входа за последний час | integer | integer | >= 0 | в момент события | оставить как контекст |
| `distance_from_usual_km` | Расстояние от обычной геолокации | float | float | 0...20000 | в момент события | оставить как контекст |
| `account_age_days` | Возраст учетной записи, дни | integer | integer | >= 0 | в момент события | оставить как контекст |
| `password_age_days` | Возраст текущего пароля, дни | integer | integer | >= 0 | в момент события | оставить как контекст |
| `known_device` | Известно ли устройство системе | binary | binary | {0, 1} | в момент события | оставить |
| `new_country` | Новая ли страна для пользователя | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `vpn_used` | Обнаружено ли использование VPN | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `analyst_verdict` | Вердикт аналитика | category | category | benign, confirmed_threat | после решения | исключить (утечка) |
| `analyst_risk_score` | Оценка риска аналитиком | integer | integer | 0...100 | после решения | исключить (утечка) |
| `is_suspicious` | Целевая метка | binary | binary | {0, 1} | после события | это `y`, в `X` не входит |

- **`y`** — целевая переменная `is_suspicious`.
- **`X`** — матрица признаков: все столбцы, кроме `y` и полей, которые исключены по причине утечки.

In [2]:
# Загрузка и первичная инвентаризация

pd.set_option('display.max_columns', None)
df = pd.read_excel(DATA_PATH, sheet_name='data')

original_shape = df.shape
original_rows, original_cols = original_shape

print(f'Загружен файл: {DATA_PATH}')

print('=== Первые 5 строк ===')
display(df.head(5))

print('\n=== Последние 5 строк ===')
display(df.tail(5))

original_rows, original_cols = df.shape
print('=== Исходный размер таблицы ===')
print(f'Число строк:   {original_rows}')
print(f'Число столбцов: {original_cols}')

print('\n=== Названия столбцов и типы данных ===')
print(df.dtypes)

mem = df.memory_usage(deep=True)
print(f'\nОбъем памяти: {mem.sum() / 1024:.3f} Кб')

# Целевой столбец присутствует ровно один раз
target_count = (df.columns == 'is_suspicious').sum()
print(f'\nСтолбцов с именем is_suspicious: {target_count}')
assert target_count == 1, 'Ожидался один столбец is_suspicious'

dict_df = pd.read_excel(DATA_PATH, sheet_name='Словарь')

# Поиск ожидаемых типов в соответствии со словарем
field_col = next(c for c in dict_df.columns if 'Поле' in c)
type_col  = next(c for c in dict_df.columns if 'Ожидаемый тип' in c)

# Словарь {поле: ожидаемый_тип}
expected_fields = (
    dict_df[[field_col, type_col]]
    .set_index(field_col)[type_col]
    .to_dict()
)

# Числовые - integer / float
numeric_cols = [col for col, t in expected_fields.items() if t in ('integer', 'float')]

print('\n=== Описательные статистики числовых признаков ===')
display(df[numeric_cols].describe().T)

# Категориальные - category
cat_cols = [col for col, t in expected_fields.items() if t == 'category' and col in df.columns]

summary = []
for col in cat_cols:
    vc = df[col].value_counts(dropna=True)
    top3 = vc.head(3)
    summary.append({
        'Столбец': col,
        'Уникальных': int(df[col].nunique(dropna=True)),
        'Топ-1': f'{top3.index[0]!r} ({top3.iloc[0]})' if len(top3) > 0 else '-',
        'Топ-2': f'{top3.index[1]!r} ({top3.iloc[1]})' if len(top3) > 1 else '-',
        'Топ-3': f'{top3.index[2]!r} ({top3.iloc[2]})' if len(top3) > 2 else '-',
    })

cat_summary = (
    pd.DataFrame(summary)
)

print('\n=== Описание категориальных признаков ===')
display(cat_summary)

Загружен файл: data/auth_events_lab01.xlsx
=== Первые 5 строк ===


,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,hour,is_weekend,failed_attempts_24h,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
0,evt_001370,2026-03-12 20:27:57,usr_046,Operations,203.0.113.178,SE,mobile,Android,Chrome,password,20,0,0,1,39.4,871,229,0,0,0,benign,65,0
1,evt_000493,2026-01-06 00:17:20,usr_068,Sales,203.0.113.132,RU,mobile,Android,Chrome,password,0,0,0,5,16.9,2383,11,1,0,0,benign,43,0
2,evt_000433,2026-01-27 17:57:11,usr_033,SOC,192.0.2.185,PL,laptop,Windows,Chrome,password,17,0,2,5,28.8,795,258,1,0,0,benign,50,0
3,evt_001361,2026-03-10 16:30:25,usr_085,Sales,192.0.2.91,NO,mobile,Android,Chrome,otp,16,0,0,3,26.1,1186,184,1,0,0,benign,40,0
4,evt_000791,2026-02-10 05:26:56,usr_092,R&D,203.0.113.190,DE,laptop,Windows,Chrome,fido2,5,0,3,6,86.0,784,182,1,0,0,benign,7,0



=== Последние 5 строк ===


,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,hour,is_weekend,failed_attempts_24h,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
1425,evt_000907,2026-03-09 00:01:20,usr_065,Operations,203.0.113.93,PL,tablet,iOS,Safari,push,0,0,0,2,49.8,3245,79,1,0,0,benign,15,0
1426,evt_001094,2026-01-17 09:15:30,usr_051,HR,192.0.2.165,RU,tablet,iOS,Safari,password,9,1,0,2,64.2,1803,239,1,0,0,benign,46,0
1427,evt_001253,2026-01-25 00:38:56,usr_061,R&D,198.51.100.128,RU,desktop,Windows,Chrome,otp,0,1,5,13,1088.3,2917,177,1,1,0,confirmed_threat,78,1
1428,evt_000743,2026-02-23 22:19:53,usr_043,HR,192.0.2.51,US,laptop,Windows,Edge,otp,22,0,1,5,14.4,1495,257,1,0,0,benign,31,0
1429,evt_000581,2026-02-14 19:29:43,usr_019,Finance,198.51.100.3,NL,laptop,Windows,Chrome,password,19,1,0,2,73.4,2915,unknown,1,0,0,benign,24,0


=== Исходный размер таблицы ===
Число строк:   1430
Число столбцов: 23

=== Названия столбцов и типы данных ===
event_id                   object
timestamp                  object
user_id                    object
department                 object
source_ip                  object
country                    object
device_type                object
os_family                  object
browser_family             object
auth_method                object
hour                        int64
is_weekend                  int64
failed_attempts_24h         int64
login_velocity_1h           int64
distance_from_usual_km    float64
account_age_days            int64
password_age_days          object
known_device                int64
new_country                 int64
vpn_used                    int64
analyst_verdict            object
analyst_risk_score          int64
is_suspicious               int64
dtype: object

Объем памяти: 1138.146 Кб

Столбцов с именем is_suspicious: 1

=== Описательные статистики 

,count,mean,std,min,25%,50%,75%,max
hour,1430.0,11.731469,7.113637,0.0,6.00,12.00,18.000,23.0
failed_attempts_24h,1430.0,1.897902,37.336751,-1.0,0.00,0.00,1.000,999.0
login_velocity_1h,1430.0,3.197902,2.222254,1.0,2.00,3.00,4.000,18.0
distance_from_usual_km,1398.0,376.536409,1853.187377,-25.0,15.70,31.35,56.275,20000.0
account_age_days,1430.0,1637.660140,921.062124,-7.0,839.75,1656.50,2365.500,3266.0
analyst_risk_score,1430.0,40.848951,23.968772,3.0,20.00,40.00,59.000,99.0



=== Описание категориальных признаков ===


,Столбец,Уникальных,Топ-1,Топ-2,Топ-3
0,user_id,120,'usr_062' (24),'usr_025' (21),'usr_082' (19)
1,department,8,'SOC' (269),'R&D' (248),'HR' (196)
2,country,9,'NO' (393),'DE' (268),'SE' (257)
3,device_type,4,'mobile' (491),'tablet' (334),'laptop' (329)
4,os_family,5,'Android' (464),'Windows' (409),'iOS' (361)
5,browser_family,5,'Chrome' (793),'Safari' (263),'Edge' (182)
6,auth_method,4,'password' (571),'push' (372),'otp' (330)
7,analyst_verdict,2,'benign' (1301),'confirmed_threat' (129),-
